# Install Necessary Packages

In [1]:
import os
import pandas as pd
import ebooklib
from ebooklib import epub
from bs4 import BeautifulSoup

## Code to read EPUB files into a plain text string for further text analysis

The EPUB files contain meta information about author, date, and title. While also conating OHCO information about Chapter and Paragraph these will all be extracted into the `books_df`.

All Books are on Box Cloud Storage

In [2]:
#File path for Epub files
directory_path = 'C:/Users/mason/Box/Text As Data Final/'

In [3]:
#Read in the Epub files and extract text
def extract_epub_to_paragraphs(epub_path):
    book = epub.read_epub(epub_path)
    title_metadata = book.get_metadata('DC', 'title')
    title = title_metadata[0][0] if title_metadata else 'Unknown Title'
    author_metadata = book.get_metadata('DC', 'creator')
    author = author_metadata[0][0] if author_metadata else 'Unknown Author'
    date_metadata = book.get_metadata('DC', 'date')
    date = date_metadata[0][0] if date_metadata else 'Unknown Date'
    print(f"Getting: {title} info...")

    paragraph_data =  []

    documents = list(book.get_items_of_type(ebooklib.ITEM_DOCUMENT))

    for chap_id, item in enumerate(documents):
        soup = BeautifulSoup(item.get_content(), 'html.parser')
        paragraphs =soup.find_all('p')

        for para_id, para in enumerate(paragraphs):
            text = para.get_text(strip=True)

            if text and len(text) >1:
                paragraph_data.append({
                    'title': title,
                    'file_path': epub_path,
                    'author': author,
                    'date': date,
                    'chapter_id': chap_id,
                    'paragraph_id': para_id,
                    'text': text

                })
    return paragraph_data

In [4]:
epub_file_location = directory_path + "SandersonFiles"
corpus_data = []

In [5]:
for filename in os.listdir(epub_file_location):
    if filename.endswith('.epub'):

        epub_path = os.path.join(epub_file_location, filename)

        try:
            book_paragraphs = extract_epub_to_paragraphs(epub_path)
            corpus_data.extend(book_paragraphs)
        except Exception as e:
            print(f"Failed to process {filename}. Error: {e}")

books_df = pd.DataFrame(corpus_data)
print("Books DataFrame created!")
print(books_df.head())


Getting: A Memory of Light info...
Getting: Arcanum Unbounded info...
Getting: Elantris info...
Getting: Isles of the Emberdark info...
Getting: Oathbringer info...
Getting: Rhythm of War (9781429952040) info...
Getting: Shadows of Self info...
Getting: Steelheart info...
Getting: The Aether of Night info...
Getting: The Alloy of Law: A Mistborn Novel info...
Getting: Sanderson, Brandon - Mistborn 06 - The Bands of Mourning info...
Getting: Mistborn: The Final Empire info...
Getting: The Hero of Ages info...
Getting: Lost Metal : A Mistborn Novel (9780765391209) info...
Getting: The Sunlit Man info...
Getting: Way of Kings info...
Getting: The Well of Ascension info...
Getting: The Wheel of Time info...
Getting: Unknown Title info...
Getting: Tress of the Emerald Sea info...
Getting: Warbreaker info...
Getting: Wind and Truth: The brand new epic Stormlight Archive novel from the international bestseller info...
Getting: Words of Radiance (Stormlight Archive, The) info...
Getting: Yumi 

## Fix titles and make sure there were no Unknowns

In [6]:
#Get each unique title and file path combination
unique_books = books_df[['title']].drop_duplicates()
print(unique_books)

                                                    title
0                                       A Memory of Light
10263                                   Arcanum Unbounded
16914                                            Elantris
23624                              Isles of the Emberdark
28256                                         Oathbringer
45694                       Rhythm of War (9781429952040)
60924                                     Shadows of Self
64884                                          Steelheart
85623                                 The Aether of Night
90525                  The Alloy of Law: A Mistborn Novel
93985   Sanderson, Brandon - Mistborn 06 - The Bands o...
99155                          Mistborn: The Final Empire
107012                                   The Hero of Ages
114868      Lost Metal : A Mistborn Novel (9780765391209)
120884                                     The Sunlit Man
124500                                       Way of Kings
137425        

In [7]:
#Map the Corrected titles to the current titles in the DataFrame
title_mapping = {
    'A Memory of Light': 'A Memory of Light',
    'Arcanum Unbounded': 'Arcanum Unbounded',
    'Elantris': 'Elantris',
    'Isles of the Emberdark': 'Isles of the Emberdark',
    'Oathbringer': 'Oathbringer',
    'Rhythm of War (9781429952040)': 'Rhythm of War',
    'Shadows of Self': 'Shadows of Self',
    'Steelheart': 'Steelheart',
    'The Aether of Night': 'The Aether of Night',
    'The Alloy of Law: A Mistborn Novel': 'The Alloy of Law',
    'Sanderson, Brandon - Mistborn 06 - The Bands of Mourning': 'The Bands of Mourning',
    'Mistborn: The Final Empire': 'The Final Empire',
    'The Hero of Ages': 'The Hero of Ages',
    'Lost Metal : A Mistborn Novel (9780765391209)': 'The Lost Metal',
    'The Sunlit Man': 'The Sunlit Man',
    'Way of Kings': 'The Way of Kings',
    'The Well of Ascension': 'The Well of Ascension',
    'The Wheel of Time': 'The Wheel of Time',
    'Unknown Title': 'Towers of Midnight',
    'Tress of the Emerald Sea': 'Tress of the Emerald Sea',
    'Warbreaker': 'Warbreaker',
    'Wind and Truth: The brand new epic Stormlight Archive novel from the international bestseller': 'Wind and Truth',
    'Words of Radiance (Stormlight Archive, The)': 'Words of Radiance',
    'Yumi and the Nightmare Painter': 'Yumi and the Nightmare Painter'
}

books_df['title'] = books_df['title'].map(title_mapping)


## Save the Resulting Brandon Sanderson Corpus to Box

In [8]:
#Save the corpus to a CSV file
books_df.to_csv(f"{directory_path}/Output/BrandonSandersonBooks.csv", index=False)
print("Books saved to CSV!")

Books saved to CSV!


In [9]:
books_df.head()

,title,file_path,author,date,chapter_id,paragraph_id,text
0,A Memory of Light,C:/Users/mason/Box/Text As Data Final/Sanderso...,Robert Jordan & Brandon Sanderson,2013-01-08,0,0,by Robert Jordan
1,A Memory of Light,C:/Users/mason/Box/Text As Data Final/Sanderso...,Robert Jordan & Brandon Sanderson,2013-01-08,0,1,The Eye of the World
2,A Memory of Light,C:/Users/mason/Box/Text As Data Final/Sanderso...,Robert Jordan & Brandon Sanderson,2013-01-08,0,2,The Great Hunt
3,A Memory of Light,C:/Users/mason/Box/Text As Data Final/Sanderso...,Robert Jordan & Brandon Sanderson,2013-01-08,0,3,The Dragon Reborn
4,A Memory of Light,C:/Users/mason/Box/Text As Data Final/Sanderso...,Robert Jordan & Brandon Sanderson,2013-01-08,0,4,The Shadow Rising


## Conclusion of this notebook

Okay so now we have a `books_df`. This dataframe contains too much OHCO and Text to be useful as a `LIB` table.